In [67]:
# ==================================================================================================
# PHASE 7 — DATASET BUILDER & QUALITY ASSURANCE
# CELL 1 — CONFIGURATION & PATH DISCOVERY
# ==================================================================================================

from pathlib import Path
import os
import json
import math
import random
import shutil
import pandas as pd
import numpy as np

# --------------------------------------------------------------------------------------------------
# PROJECT
# --------------------------------------------------------------------------------------------------

PROJECT_ROOT = Path(r"C:\LipReadingSSL")

OUTPUT_ROOT = PROJECT_ROOT / "output"

PHASE5_ROOT = OUTPUT_ROOT
PHASE6G_ROOT = OUTPUT_ROOT / "phase6_audio" / "phase6G"

DATASET_METADATA = PHASE6G_ROOT / "dataset_metadata.csv"

PHASE7_ROOT = OUTPUT_ROOT / "phase7_dataset"

TRAIN_DIR = PHASE7_ROOT / "train"
VAL_DIR   = PHASE7_ROOT / "validation"
TEST_DIR  = PHASE7_ROOT / "test"

QA_DIR = PHASE7_ROOT / "qa"

TRAIN_CSV = TRAIN_DIR / "metadata.csv"
VAL_CSV   = VAL_DIR / "metadata.csv"
TEST_CSV  = TEST_DIR / "metadata.csv"

QA_REPORT_CSV = QA_DIR / "dataset_qa.csv"
VIDEO_SUMMARY_CSV = QA_DIR / "video_summary.csv"
SPLIT_SUMMARY_CSV = QA_DIR / "split_summary.csv"

# --------------------------------------------------------------------------------------------------
# SPLIT CONFIG
# --------------------------------------------------------------------------------------------------

TRAIN_RATIO = 0.80
VAL_RATIO   = 0.10
TEST_RATIO  = 0.10

RANDOM_SEED = 42

# --------------------------------------------------------------------------------------------------
# EXPECTED DATA
# --------------------------------------------------------------------------------------------------

EXPECTED_TOTAL_CLIPS = 95674

# --------------------------------------------------------------------------------------------------
# CREATE OUTPUT DIRECTORIES
# --------------------------------------------------------------------------------------------------

for d in [
    PHASE7_ROOT,
    TRAIN_DIR,
    VAL_DIR,
    TEST_DIR,
    QA_DIR,
]:
    d.mkdir(parents=True, exist_ok=True)

# --------------------------------------------------------------------------------------------------
# HEADER
# --------------------------------------------------------------------------------------------------

print("=" * 100)
print("PHASE 7 — DATASET BUILDER & QUALITY ASSURANCE")
print("=" * 100)

print(f"Project root      : {PROJECT_ROOT}")
print(f"Phase 6G metadata : {DATASET_METADATA}")
print(f"Phase 7 output    : {PHASE7_ROOT}")

print()
print("STORAGE STRATEGY")
print("-" * 100)
print("✓ No audio files will be created")
print("✓ No WAV files will be copied")
print("✓ No mouth frames will be copied")
print("✓ Dataset uses metadata manifests")
print("✓ Existing Phase 5 / Phase 6 data are referenced by path")
print("✓ Only small CSV / JSON metadata files are generated")

print()
print("SPLIT")
print("-" * 100)
print(f"Train : {TRAIN_RATIO:.0%}")
print(f"Val   : {VAL_RATIO:.0%}")
print(f"Test  : {TEST_RATIO:.0%}")
print(f"Seed  : {RANDOM_SEED}")

# --------------------------------------------------------------------------------------------------
# PRE-CHECK
# --------------------------------------------------------------------------------------------------

print()
print("=" * 100)
print("PRE-CHECK")
print("=" * 100)

if not PROJECT_ROOT.exists():
    raise FileNotFoundError(
        f"Project root not found: {PROJECT_ROOT}"
    )

if not DATASET_METADATA.exists():
    raise FileNotFoundError(
        f"Phase 6G dataset metadata not found:\n{DATASET_METADATA}"
    )

print(f"Phase 6G metadata : FOUND")
print(f"Metadata size     : {DATASET_METADATA.stat().st_size / (1024**2):.2f} MB")

# --------------------------------------------------------------------------------------------------
# SPLIT VALIDATION
# --------------------------------------------------------------------------------------------------

if abs((TRAIN_RATIO + VAL_RATIO + TEST_RATIO) - 1.0) > 1e-9:
    raise ValueError("Train / Validation / Test ratios must sum to 1.0")

print()
print("Configuration : OK")

PHASE 7 — DATASET BUILDER & QUALITY ASSURANCE
Project root      : C:\LipReadingSSL
Phase 6G metadata : C:\LipReadingSSL\output\phase6_audio\phase6G\dataset_metadata.csv
Phase 7 output    : C:\LipReadingSSL\output\phase7_dataset

STORAGE STRATEGY
----------------------------------------------------------------------------------------------------
✓ No audio files will be created
✓ No WAV files will be copied
✓ No mouth frames will be copied
✓ Dataset uses metadata manifests
✓ Existing Phase 5 / Phase 6 data are referenced by path
✓ Only small CSV / JSON metadata files are generated

SPLIT
----------------------------------------------------------------------------------------------------
Train : 80%
Val   : 10%
Test  : 10%
Seed  : 42

PRE-CHECK
Phase 6G metadata : FOUND
Metadata size     : 16.54 MB

Configuration : OK


In [68]:
# ==================================================================================================
# PHASE 7 — CELL 2
# LOAD PHASE 6G DATASET METADATA
# ==================================================================================================

print()
print("=" * 100)
print("1. LOADING PHASE 6G DATASET METADATA")
print("=" * 100)

df = pd.read_csv(DATASET_METADATA)

print(f"Rows    : {len(df):,}")
print(f"Columns : {list(df.columns)}")

# --------------------------------------------------------------------------------------------------
# REQUIRED COLUMNS
# --------------------------------------------------------------------------------------------------

required_columns = [
    "video",
    "clip_id",
    "start_frame",
    "end_frame",
    "start_time",
    "end_time",
    "duration_sec",
    "audio_start_sample",
    "audio_end_sample",
    "transcript",
]

missing_columns = [
    c for c in required_columns
    if c not in df.columns
]

if missing_columns:
    raise ValueError(
        f"Missing required columns from Phase 6G:\n{missing_columns}"
    )

print()
print("Required columns : ALL PRESENT")

# --------------------------------------------------------------------------------------------------
# NORMALIZATION
# --------------------------------------------------------------------------------------------------

df["video"] = df["video"].astype(str).str.strip()

df["clip_id"] = pd.to_numeric(
    df["clip_id"],
    errors="coerce"
)

numeric_columns = [
    "start_frame",
    "end_frame",
    "start_time",
    "end_time",
    "duration_sec",
    "audio_start_sample",
    "audio_end_sample",
]

for col in numeric_columns:
    df[col] = pd.to_numeric(
        df[col],
        errors="coerce"
    )

df["transcript"] = (
    df["transcript"]
    .fillna("")
    .astype(str)
    .str.strip()
)

# --------------------------------------------------------------------------------------------------
# BASIC CHECKS
# --------------------------------------------------------------------------------------------------

invalid_numeric = df[
    df[
        ["clip_id"] + numeric_columns
    ].isna().any(axis=1)
]

duplicate_keys = df.duplicated(
    subset=["video", "clip_id"],
    keep=False
)

empty_transcript = df["transcript"].eq("")

print()
print("=" * 100)
print("2. INPUT VALIDATION")
print("=" * 100)

print(f"Expected clips       : {EXPECTED_TOTAL_CLIPS:,}")
print(f"Actual clips         : {len(df):,}")
print(f"Invalid numeric rows : {len(invalid_numeric):,}")
print(f"Duplicate clip keys  : {duplicate_keys.sum():,}")
print(f"Empty transcripts    : {empty_transcript.sum():,}")

if len(df) != EXPECTED_TOTAL_CLIPS:
    raise ValueError(
        f"Unexpected number of clips: {len(df):,}"
    )

if len(invalid_numeric) > 0:
    raise ValueError(
        f"Invalid numeric metadata rows: {len(invalid_numeric):,}"
    )

if duplicate_keys.sum() > 0:
    raise ValueError(
        f"Duplicate video+clip_id keys: {duplicate_keys.sum():,}"
    )

if empty_transcript.sum() > 0:
    raise ValueError(
        "Phase 6G contains empty transcripts. "
        "Phase 6G should already have filtered them."
    )

print()
print("Input validation : PASS")


1. LOADING PHASE 6G DATASET METADATA
Rows    : 95,674
Columns : ['video', 'clip_id', 'start_frame', 'end_frame', 'fps', 'start_time', 'end_time', 'duration_sec', 'audio_start_sample', 'audio_end_sample', 'audio_path', 'audio_storage_mode', 'transcript', 'status', 'error']

Required columns : ALL PRESENT

2. INPUT VALIDATION
Expected clips       : 95,674
Actual clips         : 95,674
Invalid numeric rows : 0
Duplicate clip keys  : 0
Empty transcripts    : 0

Input validation : PASS


In [69]:
# ==================================================================================================
# PHASE 7 — CELL 3
# FRAME / CLIP / SEQUENCE QUALITY ASSURANCE
# ==================================================================================================

print()
print("=" * 100)
print("3. FRAME / CLIP / SEQUENCE QUALITY ASSURANCE")
print("=" * 100)

# --------------------------------------------------------------------------------------------------
# FRAME VALIDATION
# --------------------------------------------------------------------------------------------------

frame_invalid = (
    (df["start_frame"] < 0)
    |
    (df["end_frame"] <= df["start_frame"])
)

frame_length = (
    df["end_frame"] - df["start_frame"]
)

sequence_invalid = (
    frame_length <= 0
)

# --------------------------------------------------------------------------------------------------
# EXPECTED SEQUENCE
# --------------------------------------------------------------------------------------------------

sequence_lengths = frame_length.value_counts().sort_index()

# --------------------------------------------------------------------------------------------------
# TIMESTAMP VALIDATION
# --------------------------------------------------------------------------------------------------

timestamp_invalid = (
    (df["start_time"] < 0)
    |
    (df["end_time"] <= df["start_time"])
    |
    (df["duration_sec"] <= 0)
)

# --------------------------------------------------------------------------------------------------
# AUDIO RANGE VALIDATION
# --------------------------------------------------------------------------------------------------

audio_invalid = (
    (df["audio_start_sample"] < 0)
    |
    (df["audio_end_sample"] <= df["audio_start_sample"])
)

# --------------------------------------------------------------------------------------------------
# CLIP ID VALIDATION
# --------------------------------------------------------------------------------------------------

clip_id_invalid = (
    df["clip_id"] < 0
)

# --------------------------------------------------------------------------------------------------
# TRANSCRIPT VALIDATION
# --------------------------------------------------------------------------------------------------

transcript_empty = df["transcript"].eq("")

transcript_length = df["transcript"].str.len()

transcript_invalid = transcript_empty

# --------------------------------------------------------------------------------------------------
# LABEL
# --------------------------------------------------------------------------------------------------

# Phase 7 currently uses transcript as the training label.
df["label"] = df["transcript"]

# --------------------------------------------------------------------------------------------------
# QA STATUS
# --------------------------------------------------------------------------------------------------

qa_invalid = (
    frame_invalid
    |
    sequence_invalid
    |
    timestamp_invalid
    |
    audio_invalid
    |
    clip_id_invalid
    |
    transcript_invalid
)

print(f"Total clips             : {len(df):,}")
print(f"Invalid frame rows      : {frame_invalid.sum():,}")
print(f"Invalid sequence rows   : {sequence_invalid.sum():,}")
print(f"Invalid timestamp rows  : {timestamp_invalid.sum():,}")
print(f"Invalid audio rows      : {audio_invalid.sum():,}")
print(f"Invalid clip IDs        : {clip_id_invalid.sum():,}")
print(f"Empty labels            : {transcript_empty.sum():,}")
print(f"Total QA-invalid rows   : {qa_invalid.sum():,}")

print()
print("Sequence length distribution:")

for length, count in sequence_lengths.items():
    print(f"  {int(length):>4} frames : {int(count):,}")

# --------------------------------------------------------------------------------------------------
# IMPORTANT
# --------------------------------------------------------------------------------------------------

if qa_invalid.sum() > 0:
    raise ValueError(
        f"Phase 7 input QA failed: {qa_invalid.sum():,} invalid rows."
    )

print()
print("Frame / Clip / Sequence QA : PASS")


3. FRAME / CLIP / SEQUENCE QUALITY ASSURANCE
Total clips             : 95,674
Invalid frame rows      : 0
Invalid sequence rows   : 0
Invalid timestamp rows  : 0
Invalid audio rows      : 0
Invalid clip IDs        : 0
Empty labels            : 0
Total QA-invalid rows   : 0

Sequence length distribution:
    15 frames : 95,674

Frame / Clip / Sequence QA : PASS


In [71]:
# ====================================================================================================
# PHASE 7 — CELL 4
# MOUTH FRAME AVAILABILITY QA — FAST METADATA-BASED VERSION
# ====================================================================================================

from pathlib import Path
import pandas as pd
import numpy as np
import json
import time

# ----------------------------------------------------------------------------------------------------
# CONFIG
# ----------------------------------------------------------------------------------------------------

PROJECT_ROOT = Path(r"C:\LipReadingSSL")

PHASE5_ROOT = PROJECT_ROOT / "output"

PHASE6G_METADATA = (
    PROJECT_ROOT
    / "output"
    / "phase6_audio"
    / "phase6G"
    / "dataset_metadata.csv"
)

PHASE7_ROOT = (
    PROJECT_ROOT
    / "output"
    / "phase7_dataset"
)

QA_OUTPUT = PHASE7_ROOT / "mouth_frame_qa.csv"
SUMMARY_OUTPUT = PHASE7_ROOT / "mouth_frame_qa_summary.json"

PHASE7_ROOT.mkdir(parents=True, exist_ok=True)


# ====================================================================================================
# HEADER
# ====================================================================================================

print("=" * 100)
print("PHASE 7 — CELL 4 — MOUTH FRAME AVAILABILITY QA")
print("=" * 100)

print(f"Project root       : {PROJECT_ROOT}")
print(f"Phase 6G metadata  : {PHASE6G_METADATA}")
print(f"QA output          : {QA_OUTPUT}")

print()
print("STORAGE STRATEGY")
print("-" * 100)
print("✓ No image files will be created")
print("✓ No image files will be copied")
print("✓ No recursive scan of all JPG/PNG files")
print("✓ Validation uses Phase 6G clip metadata")
print("✓ Only small QA metadata will be generated")


# ====================================================================================================
# 1. PRE-CHECK
# ====================================================================================================

print()
print("=" * 100)
print("1. PRE-CHECK")
print("=" * 100)

if not PHASE6G_METADATA.exists():
    raise FileNotFoundError(
        f"Phase 6G metadata not found:\n{PHASE6G_METADATA}"
    )

print("Phase 6G metadata : ✅ FOUND")
print(
    f"Metadata size     : "
    f"{PHASE6G_METADATA.stat().st_size / (1024**2):.2f} MB"
)


# ====================================================================================================
# 2. LOAD DATASET METADATA
# ====================================================================================================

print()
print("=" * 100)
print("2. LOADING DATASET METADATA")
print("=" * 100)

start_load = time.time()

df = pd.read_csv(PHASE6G_METADATA)

load_time = time.time() - start_load

print(f"Rows              : {len(df):,}")
print(f"Load time         : {load_time:.2f} sec")


# ====================================================================================================
# 3. REQUIRED COLUMNS
# ====================================================================================================

required_columns = [
    "video",
    "clip_id",
    "start_frame",
    "end_frame",
    "fps",
    "start_time",
    "end_time",
    "duration_sec",
    "transcript",
]

missing_columns = [
    c for c in required_columns
    if c not in df.columns
]

if missing_columns:
    raise ValueError(
        f"Missing required columns: {missing_columns}"
    )

print("Required columns  : ✅ ALL PRESENT")


# ====================================================================================================
# 4. METADATA-BASED FRAME AVAILABILITY
# ====================================================================================================

print()
print("=" * 100)
print("3. FRAME AVAILABILITY — METADATA BASED")
print("=" * 100)

print("No image files are being scanned.")
print("Checking frame ranges referenced by each clip...")


# ----------------------------------------------------------------------------------------------------
# Normalize numeric columns
# ----------------------------------------------------------------------------------------------------

numeric_columns = [
    "clip_id",
    "start_frame",
    "end_frame",
    "fps",
    "start_time",
    "end_time",
    "duration_sec",
]

for col in numeric_columns:
    df[col] = pd.to_numeric(
        df[col],
        errors="coerce"
    )


# ====================================================================================================
# 5. FRAME RANGE VALIDATION
# ====================================================================================================

invalid_numeric = (
    df[
        [
            "clip_id",
            "start_frame",
            "end_frame",
            "fps",
        ]
    ]
    .isna()
    .any(axis=1)
)

invalid_frame_range = (
    (df["start_frame"] < 0)
    |
    (df["end_frame"] <= df["start_frame"])
)

invalid_fps = (
    df["fps"] <= 0
)

invalid_frame_rows = (
    invalid_numeric
    |
    invalid_frame_range
    |
    invalid_fps
)


# ====================================================================================================
# 6. EXPECTED SEQUENCE LENGTH
# ====================================================================================================

df["_sequence_length"] = (
    df["end_frame"] - df["start_frame"]
)

invalid_sequence = (
    df["_sequence_length"] <= 0
)

expected_sequence = 15

wrong_sequence = (
    df["_sequence_length"] != expected_sequence
)


# ====================================================================================================
# 7. CLIP KEY VALIDATION
# ====================================================================================================

duplicate_clip_keys = (
    df.duplicated(
        subset=["video", "clip_id"],
        keep=False
    )
)

invalid_clip_id = (
    df["clip_id"] < 0
)


# ====================================================================================================
# 8. LABEL VALIDATION
# ====================================================================================================

transcript_series = (
    df["transcript"]
    .fillna("")
    .astype(str)
    .str.strip()
)

empty_transcript = (
    transcript_series == ""
)


# ====================================================================================================
# 9. FINAL FRAME QA STATUS
# ====================================================================================================

df["frame_status"] = np.select(
    [
        invalid_numeric,
        invalid_frame_range,
        invalid_fps,
        invalid_sequence,
        wrong_sequence,
        invalid_clip_id,
        duplicate_clip_keys,
        empty_transcript,
    ],
    [
        "INVALID_NUMERIC",
        "INVALID_FRAME_RANGE",
        "INVALID_FPS",
        "INVALID_SEQUENCE",
        "UNEXPECTED_SEQUENCE_LENGTH",
        "INVALID_CLIP_ID",
        "DUPLICATE_CLIP_KEY",
        "EMPTY_TRANSCRIPT",
    ],
    default="PASS",
)


# ====================================================================================================
# 10. SUMMARY
# ====================================================================================================

print()
print("=" * 100)
print("4. MOUTH FRAME QA SUMMARY")
print("=" * 100)

total_clips = len(df)

invalid_numeric_count = int(invalid_numeric.sum())
invalid_frame_count = int(invalid_frame_range.sum())
invalid_fps_count = int(invalid_fps.sum())
invalid_sequence_count = int(invalid_sequence.sum())
wrong_sequence_count = int(wrong_sequence.sum())
invalid_clip_id_count = int(invalid_clip_id.sum())
duplicate_clip_count = int(duplicate_clip_keys.sum())
empty_label_count = int(empty_transcript.sum())

qa_invalid = (
    df["frame_status"] != "PASS"
)

qa_invalid_count = int(qa_invalid.sum())

print(f"Total clips                    : {total_clips:,}")
print(f"Invalid numeric rows           : {invalid_numeric_count:,}")
print(f"Invalid frame ranges           : {invalid_frame_count:,}")
print(f"Invalid FPS                    : {invalid_fps_count:,}")
print(f"Invalid sequence rows          : {invalid_sequence_count:,}")
print(f"Unexpected sequence length     : {wrong_sequence_count:,}")
print(f"Invalid clip IDs               : {invalid_clip_id_count:,}")
print(f"Duplicate clip keys            : {duplicate_clip_count:,}")
print(f"Empty labels                   : {empty_label_count:,}")
print(f"Total QA-invalid rows          : {qa_invalid_count:,}")


# ====================================================================================================
# 11. SEQUENCE DISTRIBUTION
# ====================================================================================================

print()
print("Sequence length distribution:")

sequence_counts = (
    df["_sequence_length"]
    .value_counts()
    .sort_index()
)

for seq_len, count in sequence_counts.items():
    print(f"  {int(seq_len):>4} frames : {count:,}")


# ====================================================================================================
# 12. VIDEO SUMMARY
# ====================================================================================================

video_summary = []

for video, group in df.groupby("video", sort=True):

    video_invalid = (
        group["frame_status"] != "PASS"
    )

    video_summary.append({
        "video": video,
        "clips": len(group),
        "invalid_frames": int(video_invalid.sum()),
        "invalid_frame_ranges": int(
            (
                (group["start_frame"] < 0)
                |
                (group["end_frame"] <= group["start_frame"])
            ).sum()
        ),
        "invalid_sequence": int(
            (group["_sequence_length"] <= 0).sum()
        ),
        "unexpected_sequence": int(
            (group["_sequence_length"] != expected_sequence).sum()
        ),
        "duplicate_clip_keys": int(
            group.duplicated(
                subset=["video", "clip_id"],
                keep=False
            ).sum()
        ),
        "empty_transcript": int(
            group["transcript"]
            .fillna("")
            .astype(str)
            .str.strip()
            .eq("")
            .sum()
        ),
        "sequence_length": (
            int(group["_sequence_length"].mode().iloc[0])
            if not group["_sequence_length"].mode().empty
            else None
        ),
        "status": (
            "PASS"
            if not video_invalid.any()
            else "CHECK"
        ),
    })


video_summary_df = pd.DataFrame(video_summary)


# ====================================================================================================
# 13. SAVE ONLY ATTENTION ROWS
# ====================================================================================================

print()
print("=" * 100)
print("5. SAVING QA OUTPUT")
print("=" * 100)

attention_columns = [
    "video",
    "clip_id",
    "start_frame",
    "end_frame",
    "fps",
    "start_time",
    "end_time",
    "duration_sec",
    "transcript",
    "frame_status",
    "_sequence_length",
]

attention_df = df.loc[
    qa_invalid,
    attention_columns
].copy()

attention_df = attention_df.rename(
    columns={
        "_sequence_length": "sequence_length"
    }
)

attention_df.to_csv(
    QA_OUTPUT,
    index=False,
    encoding="utf-8-sig"
)

print(f"Attention rows   : {len(attention_df):,}")
print(f"QA CSV            : {QA_OUTPUT}")

if QA_OUTPUT.exists():
    print(
        f"QA CSV size       : "
        f"{QA_OUTPUT.stat().st_size / (1024**2):.2f} MB"
    )


# ====================================================================================================
# 14. SAVE SUMMARY JSON
# ====================================================================================================

summary = {
    "total_clips": total_clips,
    "invalid_numeric": invalid_numeric_count,
    "invalid_frame_ranges": invalid_frame_count,
    "invalid_fps": invalid_fps_count,
    "invalid_sequence": invalid_sequence_count,
    "unexpected_sequence_length": wrong_sequence_count,
    "invalid_clip_ids": invalid_clip_id_count,
    "duplicate_clip_keys": duplicate_clip_count,
    "empty_transcripts": empty_label_count,
    "qa_invalid_rows": qa_invalid_count,
    "expected_sequence_length": expected_sequence,
    "sequence_distribution": {
        str(int(k)): int(v)
        for k, v in sequence_counts.items()
    },
    "storage_strategy": "METADATA_ONLY",
    "image_files_scanned": 0,
    "image_files_created": 0,
    "image_files_copied": 0,
    "status": (
        "PASS"
        if qa_invalid_count == 0
        else "CHECK"
    ),
}

with open(
    SUMMARY_OUTPUT,
    "w",
    encoding="utf-8"
) as f:
    json.dump(
        summary,
        f,
        ensure_ascii=False,
        indent=2
    )


# ====================================================================================================
# 15. FINAL SUMMARY
# ====================================================================================================

print()
print("=" * 100)
print("PHASE 7 — CELL 4 FINAL SUMMARY")
print("=" * 100)

print(f"Total clips              : {total_clips:,}")
print(f"QA-invalid rows          : {qa_invalid_count:,}")
print(f"Expected sequence        : {expected_sequence} frames")

print()
print("STORAGE")
print("-" * 100)
print("Image files scanned      : 0")
print("Image files created      : 0")
print("Image files copied       : 0")

print()
print("OUTPUT")
print("-" * 100)
print(f"QA attention CSV         : {QA_OUTPUT}")
print(f"QA summary JSON          : {SUMMARY_OUTPUT}")

print()
if qa_invalid_count == 0:
    print("✅ PHASE 7 CELL 4 — MOUTH FRAME QA PASSED")
else:
    print(
        f"⚠️ PHASE 7 CELL 4 — CHECK "
        f"{qa_invalid_count:,} rows"
    )

print("=" * 100)

PHASE 7 — CELL 4 — MOUTH FRAME AVAILABILITY QA
Project root       : C:\LipReadingSSL
Phase 6G metadata  : C:\LipReadingSSL\output\phase6_audio\phase6G\dataset_metadata.csv
QA output          : C:\LipReadingSSL\output\phase7_dataset\mouth_frame_qa.csv

STORAGE STRATEGY
----------------------------------------------------------------------------------------------------
✓ No image files will be created
✓ No image files will be copied
✓ No recursive scan of all JPG/PNG files
✓ Validation uses Phase 6G clip metadata
✓ Only small QA metadata will be generated

1. PRE-CHECK
Phase 6G metadata : ✅ FOUND
Metadata size     : 16.54 MB

2. LOADING DATASET METADATA
Rows              : 95,674
Load time         : 0.40 sec
Required columns  : ✅ ALL PRESENT

3. FRAME AVAILABILITY — METADATA BASED
No image files are being scanned.
Checking frame ranges referenced by each clip...

4. MOUTH FRAME QA SUMMARY
Total clips                    : 95,674
Invalid numeric rows           : 0
Invalid frame ranges     

In [74]:
# ====================================================================================================
# PHASE 7 — CELL 5 — LABEL / DUPLICATE / DATASET QUALITY QA
# ====================================================================================================

from pathlib import Path
import json
import pandas as pd
import numpy as np
import re

print("=" * 100)
print("PHASE 7 — CELL 5 — LABEL / DUPLICATE / DATASET QUALITY QA")
print("=" * 100)

# ----------------------------------------------------------------------------------------------------
# PATHS
# ----------------------------------------------------------------------------------------------------

PROJECT_ROOT = Path(r"C:\LipReadingSSL")

INPUT_METADATA = (
    PROJECT_ROOT
    / "output"
    / "phase6_audio"
    / "phase6G"
    / "dataset_metadata.csv"
)

OUTPUT_DIR = (
    PROJECT_ROOT
    / "output"
    / "phase7_dataset"
)

QA_OUTPUT = OUTPUT_DIR / "label_duplicate_qa.csv"
SUMMARY_OUTPUT = OUTPUT_DIR / "label_duplicate_qa_summary.json"

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# ----------------------------------------------------------------------------------------------------
# CONFIG
# ----------------------------------------------------------------------------------------------------

EXPECTED_SEQUENCE_LENGTH = 15

# ----------------------------------------------------------------------------------------------------
# PRE-CHECK
# ----------------------------------------------------------------------------------------------------

print()
print("1. PRE-CHECK")
print("-" * 100)

if not INPUT_METADATA.exists():
    raise FileNotFoundError(
        f"Phase 6G metadata not found:\n{INPUT_METADATA}"
    )

print(f"Input metadata : {INPUT_METADATA}")
print(f"File size      : {INPUT_METADATA.stat().st_size / (1024**2):.2f} MB")

# ----------------------------------------------------------------------------------------------------
# LOAD
# ----------------------------------------------------------------------------------------------------

print()
print("2. LOADING DATASET METADATA")
print("-" * 100)

df = pd.read_csv(INPUT_METADATA)

print(f"Rows    : {len(df):,}")
print(f"Columns : {list(df.columns)}")

# ----------------------------------------------------------------------------------------------------
# REQUIRED COLUMNS
# ----------------------------------------------------------------------------------------------------

required_columns = [
    "video",
    "clip_id",
    "start_frame",
    "end_frame",
    "fps",
    "start_time",
    "end_time",
    "duration_sec",
    "audio_start_sample",
    "audio_end_sample",
    "audio_path",
    "audio_storage_mode",
    "transcript",
    "status",
]

missing = [c for c in required_columns if c not in df.columns]

if missing:
    raise ValueError(
        f"Missing required columns: {missing}"
    )

print("Required columns : ✅ ALL PRESENT")

# ----------------------------------------------------------------------------------------------------
# LABEL SOURCE
# ----------------------------------------------------------------------------------------------------
# IMPORTANT:
# Phase 6G does NOT contain a "label" column.
# The correct source for the spoken-text label is "transcript".
#
# We create an internal "label" column here so later Phase 7 cells
# can consistently use df["label"] without changing Phase 6G.
# ----------------------------------------------------------------------------------------------------

df["transcript"] = (
    df["transcript"]
    .fillna("")
    .astype(str)
    .str.strip()
)

df["label"] = df["transcript"]

# ----------------------------------------------------------------------------------------------------
# BASIC QUALITY
# ----------------------------------------------------------------------------------------------------

print()
print("3. LABEL QUALITY")
print("-" * 100)

empty_label = df["label"].eq("")

one_char_label = (
    df["label"].str.len() == 1
)

long_label = (
    df["label"].str.len() > 100
)

print(f"Total clips             : {len(df):,}")
print(f"Empty labels            : {empty_label.sum():,}")
print(f"1-character labels      : {one_char_label.sum():,}")
print(f">100-character labels   : {long_label.sum():,}")

# ----------------------------------------------------------------------------------------------------
# STATUS VALIDATION
# ----------------------------------------------------------------------------------------------------

print()
print("4. STATUS VALIDATION")
print("-" * 100)

status_counts = df["status"].fillna("").value_counts()

for status, count in status_counts.items():
    print(f"{str(status):20s}: {count:,}")

# Actual ASR errors must be distinguished from "empty".
actual_asr_error = (
    df["error"]
    .fillna("")
    .astype(str)
    .str.strip()
    .ne("")
)

print()
print(f"Actual ASR error rows : {actual_asr_error.sum():,}")

# ----------------------------------------------------------------------------------------------------
# CLIP KEY DUPLICATES
# ----------------------------------------------------------------------------------------------------

print()
print("5. DUPLICATE CLIP KEY VALIDATION")
print("-" * 100)

df["_clip_key"] = (
    df["video"].astype(str)
    + "::"
    + df["clip_id"].astype(str)
)

duplicate_key_mask = df["_clip_key"].duplicated(keep=False)

duplicate_key_rows = int(duplicate_key_mask.sum())
duplicate_key_groups = int(
    df.loc[duplicate_key_mask, "_clip_key"].nunique()
)

print(f"Duplicate clip rows   : {duplicate_key_rows:,}")
print(f"Duplicate key groups  : {duplicate_key_groups:,}")

# ----------------------------------------------------------------------------------------------------
# DUPLICATE TRANSCRIPT ANALYSIS
# ----------------------------------------------------------------------------------------------------
# Duplicate text is NOT considered an error.
# Sliding-window clips can legitimately have the same transcript.
# We only report the statistics.
# ----------------------------------------------------------------------------------------------------

print()
print("6. DUPLICATE TRANSCRIPT ANALYSIS")
print("-" * 100)

non_empty = df.loc[~empty_label, "label"]

label_counts = non_empty.value_counts()

duplicate_text_mask = label_counts > 1

duplicate_text_groups = int(
    duplicate_text_mask.sum()
)

duplicate_text_rows = int(
    label_counts[duplicate_text_mask].sum()
)

print(f"Rows with duplicated text : {duplicate_text_rows:,}")
print(f"Duplicated text groups    : {duplicate_text_groups:,}")
print("Duplicate text status     : INFORMATION ONLY")

# ----------------------------------------------------------------------------------------------------
# TRANSCRIPT STATISTICS
# ----------------------------------------------------------------------------------------------------

print()
print("7. LABEL STATISTICS")
print("-" * 100)

label_lengths = df["label"].str.len()

valid_lengths = label_lengths[label_lengths > 0]

if len(valid_lengths) > 0:
    print(f"Characters min    : {int(valid_lengths.min())}")
    print(f"Characters mean   : {valid_lengths.mean():.2f}")
    print(f"Characters median : {valid_lengths.median():.2f}")
    print(f"Characters max    : {int(valid_lengths.max())}")
else:
    print("No non-empty labels available.")

# Thai / whitespace token approximation.
# This is NOT a linguistic tokenizer.
token_counts = (
    df["label"]
    .str.split()
    .str.len()
)

print()
print(f"Token min    : {int(token_counts.min())}")
print(f"Token mean   : {token_counts.mean():.2f}")
print(f"Token median : {token_counts.median():.2f}")
print(f"Token max    : {int(token_counts.max())}")

# ----------------------------------------------------------------------------------------------------
# VIDEO STATISTICS
# ----------------------------------------------------------------------------------------------------

print()
print("8. VIDEO LABEL SUMMARY")
print("-" * 100)

video_summary = []

for video, g in df.groupby("video", sort=True):

    video_empty = g["label"].eq("")
    video_short = g["label"].str.len().eq(1)
    video_error = (
        g["error"]
        .fillna("")
        .astype(str)
        .str.strip()
        .ne("")
    )

    video_summary.append({
        "video": video,
        "clips": len(g),
        "empty_labels": int(video_empty.sum()),
        "one_char_labels": int(video_short.sum()),
        "asr_errors": int(video_error.sum()),
        "duplicate_clip_keys": int(
            g["_clip_key"].duplicated(keep=False).sum()
        ),
        "avg_chars": round(
            g.loc[~video_empty, "label"].str.len().mean()
            if (~video_empty).any()
            else 0,
            2
        ),
        "median_chars": round(
            g.loc[~video_empty, "label"].str.len().median()
            if (~video_empty).any()
            else 0,
            2
        ),
    })

summary_df = pd.DataFrame(video_summary)

print(summary_df.to_string(index=False))

# ----------------------------------------------------------------------------------------------------
# ATTENTION ROWS
# ----------------------------------------------------------------------------------------------------
# Only genuine issues are saved.
#
# 1-character transcript is intentionally NOT an error.
# Duplicate transcript is intentionally NOT an error.
# ----------------------------------------------------------------------------------------------------

print()
print("9. BUILDING ATTENTION CSV")
print("-" * 100)

attention_mask = (
    empty_label
    | actual_asr_error
    | duplicate_key_mask
    | long_label
)

attention = df.loc[
    attention_mask,
    [
        "video",
        "clip_id",
        "start_frame",
        "end_frame",
        "start_time",
        "end_time",
        "duration_sec",
        "transcript",
        "status",
        "error",
    ]
].copy()

# Add a compact reason field.
def get_reason(row):
    reasons = []

    if str(row["transcript"]).strip() == "":
        reasons.append("EMPTY_LABEL")

    if str(row["error"]).strip() != "":
        reasons.append("ASR_ERROR")

    if len(str(row["transcript"])) > 100:
        reasons.append("VERY_LONG_LABEL")

    key = f"{row['video']}::{row['clip_id']}"
    if key in set(
        df.loc[duplicate_key_mask, "_clip_key"]
    ):
        reasons.append("DUPLICATE_CLIP_KEY")

    return "|".join(reasons)


if len(attention) > 0:
    duplicate_keys_set = set(
        df.loc[duplicate_key_mask, "_clip_key"]
    )

    attention["validation_reason"] = attention.apply(
        lambda row: (
            "|".join([
                reason
                for reason in [
                    "EMPTY_LABEL"
                    if str(row["transcript"]).strip() == ""
                    else None,

                    "ASR_ERROR"
                    if str(row["error"]).strip() != ""
                    else None,

                    "VERY_LONG_LABEL"
                    if len(str(row["transcript"])) > 100
                    else None,

                    "DUPLICATE_CLIP_KEY"
                    if f"{row['video']}::{row['clip_id']}"
                    in duplicate_keys_set
                    else None,
                ]
                if reason is not None
            ])
        ),
        axis=1,
    )

else:
    attention["validation_reason"] = pd.Series(
        dtype=str
    )

# ----------------------------------------------------------------------------------------------------
# SAVE ATTENTION CSV
# ----------------------------------------------------------------------------------------------------

attention.to_csv(
    QA_OUTPUT,
    index=False,
    encoding="utf-8-sig"
)

# ----------------------------------------------------------------------------------------------------
# SUMMARY JSON
# ----------------------------------------------------------------------------------------------------

summary = {
    "total_clips": int(len(df)),
    "empty_labels": int(empty_label.sum()),
    "one_character_labels": int(one_char_label.sum()),
    "long_labels_gt_100": int(long_label.sum()),
    "actual_asr_errors": int(actual_asr_error.sum()),
    "duplicate_clip_key_rows": duplicate_key_rows,
    "duplicate_clip_key_groups": duplicate_key_groups,
    "duplicate_transcript_rows": duplicate_text_rows,
    "duplicate_transcript_groups": duplicate_text_groups,
    "attention_rows": int(len(attention)),
    "sequence_length_expected": EXPECTED_SEQUENCE_LENGTH,
    "duplicate_transcript_is_error": False,
    "one_character_is_error": False,
}

with open(
    SUMMARY_OUTPUT,
    "w",
    encoding="utf-8"
) as f:
    json.dump(
        summary,
        f,
        ensure_ascii=False,
        indent=2
    )

# ----------------------------------------------------------------------------------------------------
# CLEAN INTERNAL COLUMN
# ----------------------------------------------------------------------------------------------------

df.drop(columns=["_clip_key"], inplace=True)

# ----------------------------------------------------------------------------------------------------
# FINAL
# ----------------------------------------------------------------------------------------------------

print()
print("=" * 100)
print("PHASE 7 — CELL 5 FINAL SUMMARY")
print("=" * 100)

print(f"Total clips              : {len(df):,}")
print(f"Empty labels             : {empty_label.sum():,}")
print(f"1-character labels       : {one_char_label.sum():,}")
print(f"Actual ASR errors        : {actual_asr_error.sum():,}")
print(f"Duplicate clip-key rows  : {duplicate_key_rows:,}")
print(f"Duplicate transcript     : {duplicate_text_rows:,} rows")
print(f"Attention rows           : {len(attention):,}")

print()
print("IMPORTANT:")
print("  ✓ 'transcript' is used as the dataset label.")
print("  ✓ 1-character transcripts are RETAINED.")
print("  ✓ Duplicate transcript text is NOT an error.")
print("  ✓ Empty transcripts are flagged.")
print("  ✓ Actual ASR errors are checked separately.")

print()
print("OUTPUT")
print("-" * 100)
print(f"Attention CSV : {QA_OUTPUT}")
print(f"Summary JSON  : {SUMMARY_OUTPUT}")

if (
    duplicate_key_rows == 0
    and actual_asr_error.sum() == 0
):
    print()
    print("✅ PHASE 7 CELL 5 — LABEL / DUPLICATE QA PASSED")
else:
    print()
    print("⚠️ PHASE 7 CELL 5 — ATTENTION REQUIRED")

print("=" * 100)

PHASE 7 — CELL 5 — LABEL / DUPLICATE / DATASET QUALITY QA

1. PRE-CHECK
----------------------------------------------------------------------------------------------------
Input metadata : C:\LipReadingSSL\output\phase6_audio\phase6G\dataset_metadata.csv
File size      : 16.54 MB

2. LOADING DATASET METADATA
----------------------------------------------------------------------------------------------------
Rows    : 95,674
Columns : ['video', 'clip_id', 'start_frame', 'end_frame', 'fps', 'start_time', 'end_time', 'duration_sec', 'audio_start_sample', 'audio_end_sample', 'audio_path', 'audio_storage_mode', 'transcript', 'status', 'error']
Required columns : ✅ ALL PRESENT

3. LABEL QUALITY
----------------------------------------------------------------------------------------------------
Total clips             : 95,674
Empty labels            : 0
1-character labels      : 1,255
>100-character labels   : 0

4. STATUS VALIDATION
---------------------------------------------------------

In [76]:
# ==================================================================================================
# PHASE 7 — CELL 6
# TRAIN / VALIDATION / TEST SPLIT
# ==================================================================================================
# Purpose:
#   Create train / validation / test manifests from Phase 6G metadata.
#
# IMPORTANT:
#   - Uses ONLY Phase 6G metadata
#   - No image files are copied
#   - No WAV files are copied
#   - No audio is loaded
#   - No dataset_id dependency
#   - Unique key = video + clip_id
#   - Split is performed INSIDE each video using ordered clip ranges
#   - 80% Train / 10% Validation / 10% Test
#   - Seed is recorded for reproducibility
# ==================================================================================================

from pathlib import Path
import json
import time
import numpy as np
import pandas as pd


# --------------------------------------------------------------------------------------------------
# CONFIGURATION
# --------------------------------------------------------------------------------------------------

PROJECT_ROOT = Path(r"C:\LipReadingSSL")

INPUT_METADATA = (
    PROJECT_ROOT
    / "output"
    / "phase6_audio"
    / "phase6G"
    / "dataset_metadata.csv"
)

OUTPUT_DIR = (
    PROJECT_ROOT
    / "output"
    / "phase7_dataset"
)

SPLIT_METADATA = OUTPUT_DIR / "dataset_split.csv"
TRAIN_CSV = OUTPUT_DIR / "train.csv"
VAL_CSV = OUTPUT_DIR / "validation.csv"
TEST_CSV = OUTPUT_DIR / "test.csv"
SUMMARY_CSV = OUTPUT_DIR / "split_summary.csv"
SUMMARY_JSON = OUTPUT_DIR / "split_summary.json"

TRAIN_RATIO = 0.80
VAL_RATIO = 0.10
TEST_RATIO = 0.10

SEED = 42

EXPECTED_SEQUENCE_LENGTH = 15


# --------------------------------------------------------------------------------------------------
# HELPERS
# --------------------------------------------------------------------------------------------------

def section(title):
    print()
    print("=" * 100)
    print(title)
    print("=" * 100)


def fmt(n):
    return f"{int(n):,}"


# --------------------------------------------------------------------------------------------------
# START
# --------------------------------------------------------------------------------------------------

start_total = time.time()

section("PHASE 7 — CELL 6 — TRAIN / VALIDATION / TEST SPLIT")

print(f"Project root      : {PROJECT_ROOT}")
print(f"Input metadata    : {INPUT_METADATA}")
print(f"Output directory  : {OUTPUT_DIR}")

print()
print("SPLIT CONFIGURATION")
print("-" * 100)
print(f"Train             : {TRAIN_RATIO:.0%}")
print(f"Validation        : {VAL_RATIO:.0%}")
print(f"Test              : {TEST_RATIO:.0%}")
print(f"Seed              : {SEED}")
print(f"Sequence length   : {EXPECTED_SEQUENCE_LENGTH} frames")

print()
print("STORAGE STRATEGY")
print("-" * 100)
print("✓ No image files will be copied")
print("✓ No WAV files will be copied")
print("✓ No audio will be loaded")
print("✓ No new frame files will be created")
print("✓ Only metadata CSV / JSON files will be generated")


# --------------------------------------------------------------------------------------------------
# 1. PRE-CHECK
# --------------------------------------------------------------------------------------------------

section("1. PRE-CHECK")

if not PROJECT_ROOT.exists():
    raise FileNotFoundError(
        f"Project root not found:\n{PROJECT_ROOT}"
    )

if not INPUT_METADATA.exists():
    raise FileNotFoundError(
        f"Phase 6G metadata not found:\n{INPUT_METADATA}"
    )

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("Phase 6G metadata : ✅ FOUND")
print(f"Metadata size     : {INPUT_METADATA.stat().st_size / (1024 * 1024):.2f} MB")
print("Output directory  : ✅ READY")


# --------------------------------------------------------------------------------------------------
# 2. LOAD PHASE 6G METADATA
# --------------------------------------------------------------------------------------------------

section("2. LOADING PHASE 6G DATASET METADATA")

df = pd.read_csv(INPUT_METADATA)

print(f"Rows              : {fmt(len(df))}")
print(f"Columns           : {list(df.columns)}")

required_columns = [
    "video",
    "clip_id",
    "start_frame",
    "end_frame",
    "fps",
    "start_time",
    "end_time",
    "duration_sec",
    "audio_start_sample",
    "audio_end_sample",
    "audio_path",
    "audio_storage_mode",
    "transcript",
    "status",
    "error",
]

missing_columns = [
    c for c in required_columns
    if c not in df.columns
]

if missing_columns:
    raise ValueError(
        f"Missing required columns: {missing_columns}"
    )

print("Required columns : ✅ ALL PRESENT")


# --------------------------------------------------------------------------------------------------
# 3. BASIC NORMALIZATION
# --------------------------------------------------------------------------------------------------

section("3. DATA NORMALIZATION")

df["video"] = df["video"].astype(str).str.strip()
df["clip_id"] = df["clip_id"].astype(str).str.strip()
df["transcript"] = df["transcript"].fillna("").astype(str).str.strip()
df["status"] = df["status"].fillna("").astype(str).str.strip()

numeric_columns = [
    "start_frame",
    "end_frame",
    "fps",
    "start_time",
    "end_time",
    "duration_sec",
    "audio_start_sample",
    "audio_end_sample",
]

for col in numeric_columns:
    df[col] = pd.to_numeric(df[col], errors="coerce")

print("Normalization     : ✅ COMPLETE")


# --------------------------------------------------------------------------------------------------
# 4. INPUT VALIDATION
# --------------------------------------------------------------------------------------------------

section("4. INPUT VALIDATION")

duplicate_keys = df.duplicated(
    subset=["video", "clip_id"],
    keep=False
)

empty_transcript = df["transcript"].eq("")

invalid_numeric = df[
    numeric_columns
].isna().any(axis=1)

invalid_frames = (
    (df["start_frame"] < 0)
    |
    (df["end_frame"] <= df["start_frame"])
)

invalid_timestamps = (
    (df["start_time"] < 0)
    |
    (df["end_time"] <= df["start_time"])
)

invalid_sequence = (
    (df["end_frame"] - df["start_frame"])
    != EXPECTED_SEQUENCE_LENGTH
)

qa_invalid = (
    duplicate_keys
    | empty_transcript
    | invalid_numeric
    | invalid_frames
    | invalid_timestamps
    | invalid_sequence
)

print(f"Total clips              : {fmt(len(df))}")
print(f"Duplicate clip keys      : {fmt(duplicate_keys.sum())}")
print(f"Empty transcripts        : {fmt(empty_transcript.sum())}")
print(f"Invalid numeric rows     : {fmt(invalid_numeric.sum())}")
print(f"Invalid frame rows       : {fmt(invalid_frames.sum())}")
print(f"Invalid timestamp rows   : {fmt(invalid_timestamps.sum())}")
print(f"Invalid sequence rows    : {fmt(invalid_sequence.sum())}")
print(f"Total QA-invalid rows    : {fmt(qa_invalid.sum())}")

if duplicate_keys.any():
    raise ValueError(
        "Duplicate (video, clip_id) keys detected."
    )

if qa_invalid.any():
    raise ValueError(
        "Input metadata contains invalid rows. "
        "Fix Phase 6G / QA before creating the split."
    )

print("Input validation         : ✅ PASS")


# --------------------------------------------------------------------------------------------------
# 5. CREATE STABLE ORDER
# --------------------------------------------------------------------------------------------------

section("5. CREATING STABLE CLIP ORDER")

# IMPORTANT:
# We do NOT randomly distribute individual sliding-window clips.
# Clips remain ordered inside each video.
#
# This reduces leakage caused by highly overlapping neighboring windows.

df = df.sort_values(
    by=[
        "video",
        "start_frame",
        "end_frame",
        "clip_id",
    ],
    kind="mergesort"
).reset_index(drop=True)

print("Ordering               : video → start_frame → end_frame → clip_id")
print("Ordering status        : ✅ STABLE")


# --------------------------------------------------------------------------------------------------
# 6. SPLIT EACH VIDEO
# --------------------------------------------------------------------------------------------------

section("6. BUILDING TRAIN / VALIDATION / TEST SPLIT")

df["split"] = ""

split_records = []

videos = sorted(df["video"].unique())

for video in videos:

    video_mask = df["video"].eq(video)

    video_indices = df.index[video_mask].to_numpy()

    n = len(video_indices)

    if n == 0:
        continue

    # Deterministic counts.
    # Train gets the remainder so the total always equals n.
    n_train = int(np.floor(n * TRAIN_RATIO))
    n_val = int(np.floor(n * VAL_RATIO))
    n_test = n - n_train - n_val

    # Safety for very small datasets
    if n >= 3:
        n_train = max(1, n_train)
        n_val = max(1, n_val)
        n_test = max(1, n - n_train - n_val)

        # Recalculate train if rounding caused overflow.
        while n_train + n_val + n_test > n:
            n_train -= 1

    train_idx = video_indices[:n_train]
    val_idx = video_indices[n_train:n_train + n_val]
    test_idx = video_indices[n_train + n_val:]

    df.loc[train_idx, "split"] = "train"
    df.loc[val_idx, "split"] = "validation"
    df.loc[test_idx, "split"] = "test"

    split_records.append({
        "video": video,
        "total": n,
        "train": len(train_idx),
        "validation": len(val_idx),
        "test": len(test_idx),
    })


split_summary = pd.DataFrame(split_records)

print("Split method          : ORDERED / PER-VIDEO")
print("Random clip mixing    : NO")
print("Status                : ✅ COMPLETE")


# --------------------------------------------------------------------------------------------------
# 7. SPLIT COUNT VALIDATION
# --------------------------------------------------------------------------------------------------

section("7. SPLIT COUNT VALIDATION")

split_counts = (
    df["split"]
    .value_counts()
    .reindex(
        ["train", "validation", "test"],
        fill_value=0
    )
)

total_split = int(split_counts.sum())

print(f"Total clips           : {fmt(len(df))}")
print(f"Train                 : {fmt(split_counts['train'])}")
print(f"Validation            : {fmt(split_counts['validation'])}")
print(f"Test                  : {fmt(split_counts['test'])}")

if total_split != len(df):
    raise ValueError(
        "Split count mismatch."
    )

if (df["split"] == "").any():
    raise ValueError(
        "Some rows were not assigned to a split."
    )

print("Split count validation : ✅ PASS")


# --------------------------------------------------------------------------------------------------
# 8. SPLIT PERCENTAGES
# --------------------------------------------------------------------------------------------------

section("8. SPLIT PERCENTAGES")

split_percentages = (
    split_counts / len(df) * 100
)

for split_name in ["train", "validation", "test"]:
    print(
        f"{split_name:<12}: "
        f"{split_percentages[split_name]:6.2f}% "
        f"({fmt(split_counts[split_name])})"
    )


# --------------------------------------------------------------------------------------------------
# 9. LEAKAGE VALIDATION
# --------------------------------------------------------------------------------------------------

section("9. SPLIT LEAKAGE VALIDATION")

# Dataset identity is video + clip_id.
# No dataset_id column is required.

df["split_key"] = (
    df["video"].astype(str)
    + "::"
    + df["clip_id"].astype(str)
)

duplicate_split_keys = df["split_key"].duplicated(
    keep=False
)

if duplicate_split_keys.any():
    raise ValueError(
        "Duplicate split keys detected."
    )

train_keys = set(
    df.loc[df["split"] == "train", "split_key"]
)

val_keys = set(
    df.loc[df["split"] == "validation", "split_key"]
)

test_keys = set(
    df.loc[df["split"] == "test", "split_key"]
)

train_val_overlap = train_keys & val_keys
train_test_overlap = train_keys & test_keys
val_test_overlap = val_keys & test_keys

print(f"Train / Validation overlap : {len(train_val_overlap):,}")
print(f"Train / Test overlap       : {len(train_test_overlap):,}")
print(f"Validation / Test overlap  : {len(val_test_overlap):,}")

if (
    train_val_overlap
    or train_test_overlap
    or val_test_overlap
):
    raise ValueError(
        "Split leakage detected."
    )

print("Leakage validation          : ✅ PASS")


# --------------------------------------------------------------------------------------------------
# 10. VIDEO DISTRIBUTION
# --------------------------------------------------------------------------------------------------

section("10. VIDEO SPLIT DISTRIBUTION")

video_split = (
    df.groupby(
        ["video", "split"],
        sort=True
    )
    .size()
    .unstack(fill_value=0)
)

for col in ["train", "validation", "test"]:
    if col not in video_split.columns:
        video_split[col] = 0

video_split = video_split[
    ["train", "validation", "test"]
]

video_split["total"] = video_split.sum(axis=1)

print(video_split.to_string())


# --------------------------------------------------------------------------------------------------
# 11. ORDER / CONTIGUOUS SPLIT CHECK
# --------------------------------------------------------------------------------------------------

section("11. CONTIGUOUS SPLIT VALIDATION")

# Verify that each video has the expected ordering:
# train → validation → test
#
# This ensures neighboring sliding-window clips are not randomly mixed.

contiguous_errors = []

for video in videos:

    part = (
        df[df["video"] == video]
        .sort_values(
            by=["start_frame", "end_frame", "clip_id"],
            kind="mergesort"
        )
    )

    split_sequence = part["split"].tolist()

    expected_sequence = (
        ["train"] * (split_sequence.count("train"))
        +
        ["validation"] * (split_sequence.count("validation"))
        +
        ["test"] * (split_sequence.count("test"))
    )

    if split_sequence != expected_sequence:
        contiguous_errors.append(video)

print(
    f"Videos with split-order errors : "
    f"{len(contiguous_errors):,}"
)

if contiguous_errors:
    raise ValueError(
        f"Non-contiguous split detected: {contiguous_errors}"
    )

print("Contiguous split validation     : ✅ PASS")


# --------------------------------------------------------------------------------------------------
# 12. BUILD FINAL MANIFESTS
# --------------------------------------------------------------------------------------------------

section("12. BUILDING SPLIT MANIFESTS")

# Remove helper column before saving.
df = df.drop(columns=["split_key"])

train_df = df[df["split"] == "train"].copy()
val_df = df[df["split"] == "validation"].copy()
test_df = df[df["split"] == "test"].copy()

print(f"Train rows       : {fmt(len(train_df))}")
print(f"Validation rows  : {fmt(len(val_df))}")
print(f"Test rows        : {fmt(len(test_df))}")


# --------------------------------------------------------------------------------------------------
# 13. SAVE OUTPUTS
# --------------------------------------------------------------------------------------------------

section("13. SAVING METADATA")

# Master split manifest
df.to_csv(
    SPLIT_METADATA,
    index=False,
    encoding="utf-8-sig"
)

# Individual manifests
train_df.to_csv(
    TRAIN_CSV,
    index=False,
    encoding="utf-8-sig"
)

val_df.to_csv(
    VAL_CSV,
    index=False,
    encoding="utf-8-sig"
)

test_df.to_csv(
    TEST_CSV,
    index=False,
    encoding="utf-8-sig"
)

print(f"Master split CSV : {SPLIT_METADATA}")
print(f"Train CSV        : {TRAIN_CSV}")
print(f"Validation CSV   : {VAL_CSV}")
print(f"Test CSV         : {TEST_CSV}")


# --------------------------------------------------------------------------------------------------
# 14. SUMMARY TABLE
# --------------------------------------------------------------------------------------------------

section("14. SPLIT SUMMARY")

summary_rows = []

for video in videos:

    part = df[df["video"] == video]

    total = len(part)

    train_n = int((part["split"] == "train").sum())
    val_n = int((part["split"] == "validation").sum())
    test_n = int((part["split"] == "test").sum())

    summary_rows.append({
        "video": video,
        "total_clips": total,
        "train": train_n,
        "validation": val_n,
        "test": test_n,
        "train_pct": round(train_n / total * 100, 2),
        "validation_pct": round(val_n / total * 100, 2),
        "test_pct": round(test_n / total * 100, 2),
    })

global_summary = {
    "video": "TOTAL",
    "total_clips": len(df),
    "train": len(train_df),
    "validation": len(val_df),
    "test": len(test_df),
    "train_pct": round(len(train_df) / len(df) * 100, 2),
    "validation_pct": round(len(val_df) / len(df) * 100, 2),
    "test_pct": round(len(test_df) / len(df) * 100, 2),
}

summary_df = pd.DataFrame(summary_rows)

summary_df = pd.concat(
    [
        summary_df,
        pd.DataFrame([global_summary])
    ],
    ignore_index=True
)

print(summary_df.to_string(index=False))

summary_df.to_csv(
    SUMMARY_CSV,
    index=False,
    encoding="utf-8-sig"
)


# --------------------------------------------------------------------------------------------------
# 15. SUMMARY JSON
# --------------------------------------------------------------------------------------------------

summary_json = {
    "phase": "7",
    "cell": "6",
    "description": "Train / Validation / Test Split",
    "input_metadata": str(INPUT_METADATA),
    "seed": SEED,
    "split_strategy": "ordered_per_video",
    "train_ratio_target": TRAIN_RATIO,
    "validation_ratio_target": VAL_RATIO,
    "test_ratio_target": TEST_RATIO,
    "sequence_length": EXPECTED_SEQUENCE_LENGTH,
    "total_clips": int(len(df)),
    "train_clips": int(len(train_df)),
    "validation_clips": int(len(val_df)),
    "test_clips": int(len(test_df)),
    "duplicate_split_keys": int(duplicate_split_keys.sum()),
    "train_validation_overlap": int(len(train_val_overlap)),
    "train_test_overlap": int(len(train_test_overlap)),
    "validation_test_overlap": int(len(val_test_overlap)),
    "contiguous_split_errors": int(len(contiguous_errors)),
    "audio_files_created": 0,
    "image_files_created": 0,
    "status": "PASS",
}

with open(
    SUMMARY_JSON,
    "w",
    encoding="utf-8"
) as f:
    json.dump(
        summary_json,
        f,
        ensure_ascii=False,
        indent=2
    )


# --------------------------------------------------------------------------------------------------
# 16. FINAL VALIDATION
# --------------------------------------------------------------------------------------------------

section("15. FINAL VALIDATION")

checks = {
    "expected_total": len(df) == 95674,
    "train_gt_zero": len(train_df) > 0,
    "validation_gt_zero": len(val_df) > 0,
    "test_gt_zero": len(test_df) > 0,
    "no_duplicate_keys": not duplicate_split_keys.any(),
    "no_train_val_leakage": len(train_val_overlap) == 0,
    "no_train_test_leakage": len(train_test_overlap) == 0,
    "no_val_test_leakage": len(val_test_overlap) == 0,
    "contiguous_split": len(contiguous_errors) == 0,
    "all_rows_assigned": (
        len(train_df)
        + len(val_df)
        + len(test_df)
        == len(df)
    ),
}

for name, result in checks.items():
    print(
        f"{name:<28}: "
        f"{'PASS' if result else 'FAIL'}"
    )

global_status = (
    "PASS"
    if all(checks.values())
    else "CHECK"
)

print()
print(f"Global status : {global_status}")


# --------------------------------------------------------------------------------------------------
# 17. STORAGE VALIDATION
# --------------------------------------------------------------------------------------------------

section("16. STORAGE VALIDATION")

print("Image files copied : 0")
print("Image files created: 0")
print("WAV files copied   : 0")
print("WAV files created  : 0")
print("Audio loaded       : 0")

print()
print(
    f"Master CSV size    : "
    f"{SPLIT_METADATA.stat().st_size / (1024 * 1024):.2f} MB"
)

print(
    f"Train CSV size     : "
    f"{TRAIN_CSV.stat().st_size / (1024 * 1024):.2f} MB"
)

print(
    f"Validation CSV size: "
    f"{VAL_CSV.stat().st_size / (1024 * 1024):.2f} MB"
)

print(
    f"Test CSV size      : "
    f"{TEST_CSV.stat().st_size / (1024 * 1024):.2f} MB"
)


# --------------------------------------------------------------------------------------------------
# FINAL
# --------------------------------------------------------------------------------------------------

elapsed = time.time() - start_total

section("PHASE 7 — CELL 6 FINAL SUMMARY")

print(f"Total clips          : {fmt(len(df))}")
print(f"Train clips          : {fmt(len(train_df))}")
print(f"Validation clips     : {fmt(len(val_df))}")
print(f"Test clips           : {fmt(len(test_df))}")

print()
print("Split strategy       : ORDERED / PER-VIDEO")
print("Unique key           : video + clip_id")
print("Sequence length      : 15 frames")
print("Seed                 : 42")

print()
print(f"Duplicate keys       : {fmt(int(duplicate_split_keys.sum()))}")
print(f"Leakage errors       : {fmt(len(train_val_overlap) + len(train_test_overlap) + len(val_test_overlap))}")
print(f"Contiguous errors    : {fmt(len(contiguous_errors))}")

print()
print("STORAGE")
print("-" * 100)
print("Image files created  : 0")
print("WAV files created    : 0")
print("Audio loaded         : 0")

print()
print("OUTPUT")
print("-" * 100)
print(f"Master split CSV     : {SPLIT_METADATA}")
print(f"Train CSV            : {TRAIN_CSV}")
print(f"Validation CSV       : {VAL_CSV}")
print(f"Test CSV              : {TEST_CSV}")
print(f"Summary CSV          : {SUMMARY_CSV}")
print(f"Summary JSON         : {SUMMARY_JSON}")

print()
print(f"Elapsed time         : {elapsed:.2f} sec")

if global_status == "PASS":
    print()
    print("🎉 PHASE 7 CELL 6 — TRAIN / VALIDATION / TEST SPLIT PASSED")
else:
    print()
    print("⚠️ PHASE 7 CELL 6 — SPLIT REQUIRES CHECK")

print("=" * 100)


PHASE 7 — CELL 6 — TRAIN / VALIDATION / TEST SPLIT
Project root      : C:\LipReadingSSL
Input metadata    : C:\LipReadingSSL\output\phase6_audio\phase6G\dataset_metadata.csv
Output directory  : C:\LipReadingSSL\output\phase7_dataset

SPLIT CONFIGURATION
----------------------------------------------------------------------------------------------------
Train             : 80%
Validation        : 10%
Test              : 10%
Seed              : 42
Sequence length   : 15 frames

STORAGE STRATEGY
----------------------------------------------------------------------------------------------------
✓ No image files will be copied
✓ No WAV files will be copied
✓ No audio will be loaded
✓ No new frame files will be created
✓ Only metadata CSV / JSON files will be generated

1. PRE-CHECK
Phase 6G metadata : ✅ FOUND
Metadata size     : 16.54 MB
Output directory  : ✅ READY

2. LOADING PHASE 6G DATASET METADATA
Rows              : 95,674
Columns           : ['video', 'clip_id', 'start_frame', 'end

In [78]:
# ==================================================================================================
# PHASE 7 — CELL 7 — DATASET MANIFEST BUILDER & FINAL QA
# ==================================================================================================

from pathlib import Path
import json
import time
import pandas as pd
import numpy as np

# --------------------------------------------------------------------------------------------------
# CONFIG
# --------------------------------------------------------------------------------------------------

PROJECT_ROOT = Path(r"C:\LipReadingSSL")

PHASE7_DIR = PROJECT_ROOT / "output" / "phase7_dataset"

INPUT_METADATA = (
    PROJECT_ROOT
    / "output"
    / "phase6_audio"
    / "phase6G"
    / "dataset_metadata.csv"
)

INPUT_SPLIT = PHASE7_DIR / "dataset_split.csv"

TRAIN_OUTPUT = PHASE7_DIR / "train_manifest.csv"
VAL_OUTPUT   = PHASE7_DIR / "validation_manifest.csv"
TEST_OUTPUT  = PHASE7_DIR / "test_manifest.csv"

MASTER_OUTPUT = PHASE7_DIR / "dataset_manifest.csv"
SUMMARY_OUTPUT = PHASE7_DIR / "dataset_manifest_summary.csv"
QA_OUTPUT = PHASE7_DIR / "dataset_manifest_qa.csv"
JSON_OUTPUT = PHASE7_DIR / "dataset_manifest_summary.json"

EXPECTED_SEQUENCE_LENGTH = 15

REQUIRED_METADATA_COLUMNS = [
    "video",
    "clip_id",
    "start_frame",
    "end_frame",
    "fps",
    "start_time",
    "end_time",
    "duration_sec",
    "audio_start_sample",
    "audio_end_sample",
    "audio_path",
    "audio_storage_mode",
    "transcript",
    "status",
    "error",
]

REQUIRED_SPLIT_COLUMNS = [
    "video",
    "clip_id",
    "split",
]

SPLIT_ORDER = ["train", "validation", "test"]

start_time_total = time.time()


# --------------------------------------------------------------------------------------------------
# HELPERS
# --------------------------------------------------------------------------------------------------

def section(title):
    print()
    print("=" * 100)
    print(title)
    print("=" * 100)


def status_line(name, value):
    print(f"{name:<30}: {value}")


def file_size_mb(path):
    if path.exists():
        return path.stat().st_size / (1024 * 1024)
    return 0.0


# --------------------------------------------------------------------------------------------------
# HEADER
# --------------------------------------------------------------------------------------------------

section("PHASE 7 — CELL 7 — DATASET MANIFEST BUILDER & FINAL QA")

print(f"Project root      : {PROJECT_ROOT}")
print(f"Phase 6G metadata : {INPUT_METADATA}")
print(f"Split metadata    : {INPUT_SPLIT}")
print(f"Output directory  : {PHASE7_DIR}")

print()
print("STORAGE STRATEGY")
print("-" * 100)
print("✓ No image files will be created")
print("✓ No image files will be copied")
print("✓ No WAV files will be created")
print("✓ No WAV files will be copied")
print("✓ No audio will be loaded")
print("✓ Dataset uses metadata references only")
print("✓ transcript is used as the training label")
print("✓ Full WAV path + sample range are retained")


# ==================================================================================================
# 1. PRE-CHECK
# ==================================================================================================

section("1. PRE-CHECK")

if not INPUT_METADATA.exists():
    raise FileNotFoundError(
        f"Phase 6G metadata not found:\n{INPUT_METADATA}"
    )

if not INPUT_SPLIT.exists():
    raise FileNotFoundError(
        f"Phase 7 Cell 6 split metadata not found:\n{INPUT_SPLIT}"
    )

PHASE7_DIR.mkdir(parents=True, exist_ok=True)

status_line(
    "Phase 6G metadata",
    f"FOUND ({file_size_mb(INPUT_METADATA):.2f} MB)"
)

status_line(
    "Split metadata",
    f"FOUND ({file_size_mb(INPUT_SPLIT):.2f} MB)"
)

status_line("Output directory", "READY")


# ==================================================================================================
# 2. LOAD PHASE 6G METADATA
# ==================================================================================================

section("2. LOADING PHASE 6G METADATA")

df_meta = pd.read_csv(INPUT_METADATA)

print(f"Rows    : {len(df_meta):,}")
print(f"Columns : {list(df_meta.columns)}")

missing_meta = [
    c for c in REQUIRED_METADATA_COLUMNS
    if c not in df_meta.columns
]

if missing_meta:
    raise ValueError(
        f"Missing Phase 6G columns: {missing_meta}"
    )

print("Required columns : ✅ ALL PRESENT")


# ==================================================================================================
# 3. LOAD SPLIT METADATA
# ==================================================================================================

section("3. LOADING CELL 6 SPLIT METADATA")

df_split = pd.read_csv(INPUT_SPLIT)

print(f"Rows    : {len(df_split):,}")
print(f"Columns : {list(df_split.columns)}")

missing_split = [
    c for c in REQUIRED_SPLIT_COLUMNS
    if c not in df_split.columns
]

if missing_split:
    raise ValueError(
        f"Missing split columns: {missing_split}"
    )

print("Required columns : ✅ ALL PRESENT")


# ==================================================================================================
# 4. NORMALIZATION
# ==================================================================================================

section("4. DATA NORMALIZATION")

for df in [df_meta, df_split]:

    df["video"] = df["video"].astype(str).str.strip()
    df["clip_id"] = df["clip_id"].astype(str).str.strip()

df_meta["transcript"] = (
    df_meta["transcript"]
    .fillna("")
    .astype(str)
    .str.strip()
)

df_meta["status"] = (
    df_meta["status"]
    .fillna("")
    .astype(str)
    .str.strip()
)

df_meta["error"] = (
    df_meta["error"]
    .fillna("")
    .astype(str)
    .str.strip()
)

df_split["split"] = (
    df_split["split"]
    .fillna("")
    .astype(str)
    .str.strip()
    .str.lower()
)

print("Normalization : ✅ COMPLETE")


# ==================================================================================================
# 5. BUILD STABLE CLIP KEY
# ==================================================================================================

section("5. BUILDING STABLE CLIP KEY")

df_meta["clip_key"] = (
    df_meta["video"]
    + "::"
    + df_meta["clip_id"]
)

df_split["clip_key"] = (
    df_split["video"]
    + "::"
    + df_split["clip_id"]
)

meta_duplicate_keys = int(
    df_meta["clip_key"].duplicated().sum()
)

split_duplicate_keys = int(
    df_split["clip_key"].duplicated().sum()
)

print(f"Phase 6G duplicate keys : {meta_duplicate_keys:,}")
print(f"Split duplicate keys    : {split_duplicate_keys:,}")

if meta_duplicate_keys > 0:
    raise ValueError("Duplicate clip keys detected in Phase 6G metadata.")

if split_duplicate_keys > 0:
    raise ValueError("Duplicate clip keys detected in split metadata.")

print("Clip key validation : ✅ PASS")


# ==================================================================================================
# 6. SPLIT VALIDATION
# ==================================================================================================

section("6. SPLIT VALIDATION")

invalid_split_values = sorted(
    set(df_split["split"]) - set(SPLIT_ORDER)
)

if invalid_split_values:
    raise ValueError(
        f"Invalid split values: {invalid_split_values}"
    )

missing_in_split = set(df_meta["clip_key"]) - set(df_split["clip_key"])
extra_in_split = set(df_split["clip_key"]) - set(df_meta["clip_key"])

print(f"Metadata clips          : {len(df_meta):,}")
print(f"Split clips             : {len(df_split):,}")
print(f"Missing from split      : {len(missing_in_split):,}")
print(f"Extra split clips       : {len(extra_in_split):,}")

if missing_in_split or extra_in_split:
    raise ValueError(
        "Phase 6G metadata and Cell 6 split metadata do not contain "
        "the same clip keys."
    )

print("Split coverage : ✅ PASS")


# ==================================================================================================
# 7. MERGE METADATA + SPLIT
# ==================================================================================================

section("7. MERGING DATASET METADATA + SPLIT")

split_lookup = df_split[
    [
        "clip_key",
        "split",
    ]
].copy()

manifest = df_meta.merge(
    split_lookup,
    on="clip_key",
    how="left",
    validate="one_to_one",
)

print(f"Merged rows : {len(manifest):,}")

if len(manifest) != len(df_meta):
    raise ValueError("Merge changed the number of dataset rows.")

if manifest["split"].isna().any():
    raise ValueError("Some clips do not have a split assignment.")

print("Merge : ✅ PASS")


# ==================================================================================================
# 8. CREATE DATASET ID
# ==================================================================================================

section("8. BUILDING DATASET IDENTIFIER")

# IMPORTANT:
# dataset_id is CREATED HERE.
# It is NOT expected to exist in Phase 6G.

manifest["dataset_id"] = (
    manifest["video"]
    + "_"
    + manifest["clip_id"]
)

dataset_id_duplicates = int(
    manifest["dataset_id"].duplicated().sum()
)

print(f"Dataset IDs : {manifest['dataset_id'].nunique():,}")
print(f"Duplicate dataset IDs : {dataset_id_duplicates:,}")

if dataset_id_duplicates > 0:
    raise ValueError("Duplicate dataset_id detected.")

print("Dataset ID validation : ✅ PASS")


# ==================================================================================================
# 9. LABEL CREATION
# ==================================================================================================

section("9. BUILDING TRAINING LABEL")

# transcript is the actual label.
# Do NOT expect a separate "label" column.

manifest["label"] = manifest["transcript"]

empty_labels = int(
    manifest["label"].eq("").sum()
)

one_char_labels = int(
    manifest["label"].str.len().eq(1).sum()
)

long_labels = int(
    manifest["label"].str.len().gt(100).sum()
)

print(f"Empty labels            : {empty_labels:,}")
print(f"1-character labels      : {one_char_labels:,}")
print(f">100-character labels   : {long_labels:,}")

if empty_labels > 0:
    raise ValueError(
        f"Dataset contains {empty_labels:,} empty labels."
    )

print("Label validation : ✅ PASS")


# ==================================================================================================
# 10. FRAME / SEQUENCE VALIDATION
# ==================================================================================================

section("10. FRAME / SEQUENCE VALIDATION")

numeric_columns = [
    "start_frame",
    "end_frame",
    "fps",
    "start_time",
    "end_time",
    "duration_sec",
    "audio_start_sample",
    "audio_end_sample",
]

for col in numeric_columns:
    manifest[col] = pd.to_numeric(
        manifest[col],
        errors="coerce"
    )

numeric_invalid = int(
    manifest[numeric_columns].isna().any(axis=1).sum()
)

frame_invalid = int(
    (
        (manifest["start_frame"] < 0)
        |
        (manifest["end_frame"] <= manifest["start_frame"])
    ).sum()
)

fps_invalid = int(
    (manifest["fps"] <= 0).sum()
)

timestamp_invalid = int(
    (
        (manifest["start_time"] < 0)
        |
        (manifest["end_time"] <= manifest["start_time"])
    ).sum()
)

audio_invalid = int(
    (
        (manifest["audio_start_sample"] < 0)
        |
        (manifest["audio_end_sample"] <= manifest["audio_start_sample"])
    ).sum()
)

sequence_length = (
    manifest["end_frame"]
    - manifest["start_frame"]
)

sequence_invalid = int(
    (sequence_length != EXPECTED_SEQUENCE_LENGTH).sum()
)

print(f"Numeric invalid rows       : {numeric_invalid:,}")
print(f"Invalid frame rows         : {frame_invalid:,}")
print(f"Invalid FPS rows           : {fps_invalid:,}")
print(f"Invalid timestamp rows     : {timestamp_invalid:,}")
print(f"Invalid audio rows         : {audio_invalid:,}")
print(f"Invalid sequence rows      : {sequence_invalid:,}")

total_alignment_invalid = (
    numeric_invalid
    + frame_invalid
    + fps_invalid
    + timestamp_invalid
    + audio_invalid
    + sequence_invalid
)

print(f"Total frame/audio invalid  : {total_alignment_invalid:,}")

if total_alignment_invalid > 0:
    raise ValueError(
        "Frame / timestamp / audio / sequence validation failed."
    )

manifest["sequence_length"] = sequence_length.astype(int)

print("Frame / sequence validation : ✅ PASS")


# ==================================================================================================
# 11. AUDIO REFERENCE VALIDATION
# ==================================================================================================

section("11. FULL WAV REFERENCE VALIDATION")

missing_audio_path = int(
    manifest["audio_path"]
    .fillna("")
    .astype(str)
    .str.strip()
    .eq("")
    .sum()
)

storage_modes = (
    manifest["audio_storage_mode"]
    .fillna("")
    .astype(str)
    .str.strip()
    .value_counts()
)

print(f"Missing audio path : {missing_audio_path:,}")
print("Audio storage modes:")

for mode, count in storage_modes.items():
    print(f"  {mode:<24} : {count:,}")

if missing_audio_path > 0:
    raise ValueError(
        "Some dataset rows do not contain an audio reference."
    )

print("Audio reference : ✅ PASS")
print("Audio files loaded : 0")


# ==================================================================================================
# 12. STATUS VALIDATION
# ==================================================================================================

section("12. STATUS VALIDATION")

status_counts = (
    manifest["status"]
    .value_counts()
)

for status, count in status_counts.items():
    print(f"{status:<20}: {count:,}")

actual_asr_errors = int(
    manifest["error"].ne("").sum()
)

non_ok_status = int(
    (~manifest["status"].eq("ok")).sum()
)

print(f"Actual ASR error rows : {actual_asr_errors:,}")
print(f"Non-ok status rows    : {non_ok_status:,}")

if actual_asr_errors > 0:
    raise ValueError(
        f"Dataset contains {actual_asr_errors:,} actual ASR error rows."
    )

if non_ok_status > 0:
    raise ValueError(
        f"Dataset contains {non_ok_status:,} non-ok status rows."
    )

print("Status validation : ✅ PASS")


# ==================================================================================================
# 13. DUPLICATE TRANSCRIPT INFORMATION
# ==================================================================================================

section("13. DUPLICATE TRANSCRIPT INFORMATION")

duplicate_text_mask = (
    manifest["transcript"]
    .duplicated(keep=False)
)

duplicate_text_rows = int(
    duplicate_text_mask.sum()
)

duplicate_text_groups = int(
    manifest.loc[
        duplicate_text_mask,
        "transcript"
    ].nunique()
)

print(f"Rows with duplicated transcript : {duplicate_text_rows:,}")
print(f"Duplicated transcript groups    : {duplicate_text_groups:,}")

print()
print("NOTE:")
print("Duplicate transcript text is INFORMATION ONLY.")
print("It is NOT removed because sliding-window clips can legitimately")
print("contain identical transcripts.")


# ==================================================================================================
# 14. BUILD FINAL MANIFEST COLUMNS
# ==================================================================================================

section("14. BUILDING FINAL DATASET MANIFEST")

manifest_columns = [
    "dataset_id",
    "video",
    "clip_id",
    "split",

    "start_frame",
    "end_frame",
    "sequence_length",
    "fps",

    "start_time",
    "end_time",
    "duration_sec",

    "audio_start_sample",
    "audio_end_sample",
    "audio_path",
    "audio_storage_mode",

    "transcript",
    "label",

    "status",
    "error",
]

manifest = manifest[manifest_columns].copy()

manifest = manifest.sort_values(
    [
        "split",
        "video",
        "start_frame",
        "end_frame",
        "clip_id",
    ]
).reset_index(drop=True)

print(f"Final manifest rows    : {len(manifest):,}")
print(f"Final manifest columns : {len(manifest.columns)}")
print("Manifest build : ✅ COMPLETE")


# ==================================================================================================
# 15. SPLIT COUNTS
# ==================================================================================================

section("15. FINAL SPLIT COUNTS")

split_counts = (
    manifest["split"]
    .value_counts()
    .reindex(SPLIT_ORDER, fill_value=0)
)

for split_name, count in split_counts.items():
    pct = (
        count / len(manifest) * 100
        if len(manifest)
        else 0
    )

    print(
        f"{split_name:<12}: "
        f"{count:>8,} "
        f"({pct:6.2f}%)"
    )

if int(split_counts.sum()) != len(manifest):
    raise ValueError("Split counts do not sum to total dataset size.")

print("Split counts : ✅ PASS")


# ==================================================================================================
# 16. VIDEO DISTRIBUTION
# ==================================================================================================

section("16. VIDEO DISTRIBUTION")

video_split = pd.crosstab(
    manifest["video"],
    manifest["split"]
).reindex(
    columns=SPLIT_ORDER,
    fill_value=0
)

video_split["total"] = video_split.sum(axis=1)

print(video_split.to_string())

print()
print("Video distribution : ✅ COMPLETE")


# ==================================================================================================
# 17. LEAKAGE CHECK
# ==================================================================================================

section("17. SPLIT LEAKAGE VALIDATION")

train_ids = set(
    manifest.loc[
        manifest["split"] == "train",
        "dataset_id"
    ]
)

val_ids = set(
    manifest.loc[
        manifest["split"] == "validation",
        "dataset_id"
    ]
)

test_ids = set(
    manifest.loc[
        manifest["split"] == "test",
        "dataset_id"
    ]
)

train_val_overlap = len(train_ids & val_ids)
train_test_overlap = len(train_ids & test_ids)
val_test_overlap = len(val_ids & test_ids)

print(f"Train / Validation overlap : {train_val_overlap:,}")
print(f"Train / Test overlap       : {train_test_overlap:,}")
print(f"Validation / Test overlap  : {val_test_overlap:,}")

if (
    train_val_overlap
    or train_test_overlap
    or val_test_overlap
):
    raise ValueError(
        "Dataset split leakage detected."
    )

print("Split leakage : ✅ PASS")


# ==================================================================================================
# 18. FINAL QA TABLE
# ==================================================================================================

section("18. FINAL DATASET QA")

qa_rows = []

for split_name in SPLIT_ORDER:

    part = manifest[
        manifest["split"] == split_name
    ]

    qa_rows.append({
        "split": split_name,
        "clips": len(part),
        "empty_labels": int(part["label"].eq("").sum()),
        "one_char_labels": int(
            part["label"].str.len().eq(1).sum()
        ),
        "duplicate_clip_keys": int(
            part["video"].astype(str)
            .str.cat(
                part["clip_id"].astype(str),
                sep="::"
            )
            .duplicated()
            .sum()
        ),
        "sequence_invalid": int(
            (part["sequence_length"] != EXPECTED_SEQUENCE_LENGTH).sum()
        ),
        "audio_reference_missing": int(
            part["audio_path"]
            .fillna("")
            .astype(str)
            .str.strip()
            .eq("")
            .sum()
        ),
    })

qa_df = pd.DataFrame(qa_rows)

print(qa_df.to_string(index=False))


# ==================================================================================================
# 19. SAVE MANIFESTS
# ==================================================================================================

section("19. SAVING DATASET MANIFESTS")

manifest.to_csv(
    MASTER_OUTPUT,
    index=False,
    encoding="utf-8-sig"
)

manifest[
    manifest["split"] == "train"
].to_csv(
    TRAIN_OUTPUT,
    index=False,
    encoding="utf-8-sig"
)

manifest[
    manifest["split"] == "validation"
].to_csv(
    VAL_OUTPUT,
    index=False,
    encoding="utf-8-sig"
)

manifest[
    manifest["split"] == "test"
].to_csv(
    TEST_OUTPUT,
    index=False,
    encoding="utf-8-sig"
)

qa_df.to_csv(
    QA_OUTPUT,
    index=False,
    encoding="utf-8-sig"
)

print(f"Master manifest : {MASTER_OUTPUT}")
print(f"Train manifest  : {TRAIN_OUTPUT}")
print(f"Val manifest    : {VAL_OUTPUT}")
print(f"Test manifest   : {TEST_OUTPUT}")
print(f"QA CSV          : {QA_OUTPUT}")


# ==================================================================================================
# 20. SUMMARY
# ==================================================================================================

section("20. DATASET SUMMARY")

summary_rows = []

for video_name, group in manifest.groupby("video"):

    summary_rows.append({
        "video": video_name,
        "clips": len(group),
        "train": int((group["split"] == "train").sum()),
        "validation": int((group["split"] == "validation").sum()),
        "test": int((group["split"] == "test").sum()),
        "avg_chars": round(
            group["label"].str.len().mean(),
            2
        ),
        "median_chars": round(
            group["label"].str.len().median(),
            2
        ),
        "one_char_labels": int(
            group["label"].str.len().eq(1).sum()
        ),
        "sequence_length": int(
            group["sequence_length"].mode().iloc[0]
        ),
    })

summary_df = pd.DataFrame(summary_rows)

summary_df.to_csv(
    SUMMARY_OUTPUT,
    index=False,
    encoding="utf-8-sig"
)

print(summary_df.to_string(index=False))


# ==================================================================================================
# 21. GLOBAL VALIDATION
# ==================================================================================================

section("21. GLOBAL VALIDATION")

checks = {
    "total_rows_gt_zero":
        len(manifest) > 0,

    "expected_total":
        len(manifest) == len(df_meta),

    "no_empty_labels":
        empty_labels == 0,

    "no_duplicate_dataset_ids":
        dataset_id_duplicates == 0,

    "no_duplicate_clip_keys":
        meta_duplicate_keys == 0,

    "no_alignment_errors":
        total_alignment_invalid == 0,

    "no_asr_errors":
        actual_asr_errors == 0,

    "all_split_values_valid":
        not invalid_split_values,

    "all_rows_have_split":
        manifest["split"].notna().all(),

    "no_train_val_leakage":
        train_val_overlap == 0,

    "no_train_test_leakage":
        train_test_overlap == 0,

    "no_val_test_leakage":
        val_test_overlap == 0,

    "all_sequence_length_15":
        (manifest["sequence_length"] == EXPECTED_SEQUENCE_LENGTH).all(),

    "no_missing_audio_reference":
        missing_audio_path == 0,
}

for name, passed in checks.items():
    print(
        f"{name:<35}: "
        f"{'PASS' if passed else 'FAIL'}"
    )

global_pass = all(checks.values())


# ==================================================================================================
# 22. SAVE JSON SUMMARY
# ==================================================================================================

summary_json = {
    "phase": "7",
    "cell": "7",
    "total_clips": int(len(manifest)),
    "train_clips": int(split_counts["train"]),
    "validation_clips": int(split_counts["validation"]),
    "test_clips": int(split_counts["test"]),
    "sequence_length": EXPECTED_SEQUENCE_LENGTH,
    "empty_labels": empty_labels,
    "one_character_labels": one_char_labels,
    "actual_asr_errors": actual_asr_errors,
    "duplicate_dataset_ids": dataset_id_duplicates,
    "duplicate_transcript_rows": duplicate_text_rows,
    "alignment_invalid": total_alignment_invalid,
    "audio_files_created": 0,
    "audio_files_copied": 0,
    "audio_loaded": 0,
    "image_files_created": 0,
    "image_files_copied": 0,
    "split_leakage": {
        "train_validation": train_val_overlap,
        "train_test": train_test_overlap,
        "validation_test": val_test_overlap,
    },
    "global_status": "PASS" if global_pass else "CHECK",
}

with open(
    JSON_OUTPUT,
    "w",
    encoding="utf-8"
) as f:
    json.dump(
        summary_json,
        f,
        ensure_ascii=False,
        indent=2
    )


# ==================================================================================================
# 23. FINAL OUTPUT
# ==================================================================================================

elapsed = time.time() - start_time_total

section("PHASE 7 — CELL 7 FINAL SUMMARY")

print(f"Total clips             : {len(manifest):,}")
print(f"Train clips             : {split_counts['train']:,}")
print(f"Validation clips        : {split_counts['validation']:,}")
print(f"Test clips              : {split_counts['test']:,}")
print()
print(f"Empty labels            : {empty_labels:,}")
print(f"1-character labels      : {one_char_labels:,}")
print(f"Actual ASR errors       : {actual_asr_errors:,}")
print(f"Alignment errors        : {total_alignment_invalid:,}")
print(f"Duplicate dataset IDs   : {dataset_id_duplicates:,}")
print(f"Duplicate transcript    : {duplicate_text_rows:,} rows")
print()
print(f"Sequence length         : {EXPECTED_SEQUENCE_LENGTH} frames")
print("Label source            : transcript")
print("Audio strategy          : FULL_WAV_REFERENCE")
print()
print("STORAGE")
print("-" * 100)
print("Image files created     : 0")
print("Image files copied      : 0")
print("WAV files created       : 0")
print("WAV files copied        : 0")
print("Audio loaded            : 0")
print()
print("OUTPUT")
print("-" * 100)
print(f"Master manifest         : {MASTER_OUTPUT}")
print(f"Train manifest          : {TRAIN_OUTPUT}")
print(f"Validation manifest     : {VAL_OUTPUT}")
print(f"Test manifest           : {TEST_OUTPUT}")
print(f"QA CSV                  : {QA_OUTPUT}")
print(f"Summary CSV             : {SUMMARY_OUTPUT}")
print(f"Summary JSON            : {JSON_OUTPUT}")
print()
print(f"Elapsed time            : {elapsed:.2f} sec")

if global_pass:
    print()
    print("🎉 PHASE 7 — CELL 7 PASSED")
    print("Dataset manifest is ready for the next dataset/training stage.")
else:
    print()
    print("⚠️ PHASE 7 — CELL 7 REQUIRES CHECK")
    print("Review the validation results above.")

print("=" * 100)


PHASE 7 — CELL 7 — DATASET MANIFEST BUILDER & FINAL QA
Project root      : C:\LipReadingSSL
Phase 6G metadata : C:\LipReadingSSL\output\phase6_audio\phase6G\dataset_metadata.csv
Split metadata    : C:\LipReadingSSL\output\phase7_dataset\dataset_split.csv
Output directory  : C:\LipReadingSSL\output\phase7_dataset

STORAGE STRATEGY
----------------------------------------------------------------------------------------------------
✓ No image files will be created
✓ No image files will be copied
✓ No WAV files will be created
✓ No WAV files will be copied
✓ No audio will be loaded
✓ Dataset uses metadata references only
✓ transcript is used as the training label
✓ Full WAV path + sample range are retained

1. PRE-CHECK
Phase 6G metadata             : FOUND (16.54 MB)
Split metadata                : FOUND (17.13 MB)
Output directory              : READY

2. LOADING PHASE 6G METADATA
Rows    : 95,674
Columns : ['video', 'clip_id', 'start_frame', 'end_frame', 'fps', 'start_time', 'end_time'